# LLM from scratch · the series notebook (Transformers from scratch)

One notebook for the whole YouTube series: GPT-2 small, opened up step by step, until we train our own GPT-2-style models and the
modern variants. A new chapter is added with every episode.

**How it works**
- Run **Setup** first, every time you open the notebook.
- Each chapter needs only Setup plus the **toolkit** cells of earlier chapters (the last cell of each chapter), so you can jump
  straight to any chapter.
- Cells marked **✏️ YOUR TURN** have blanks (`___`). Fill them in and run; the check cell below prints ✅ when you're right.
  "Run all" stops at the first blank; that's expected. Solutions are at the end of each chapter.

**Chapters**
1. Chapter 1 · Text → numbers → vectors

## Setup
Loads GPT-2 small (about 500 MB, downloaded once per session). Training chapters will save datasets and checkpoints so a
disconnected session can resume: set `USE_DRIVE = True` there, and Colab will ask you to sign in to Google and allow access to your
Drive. Everything goes into one folder, `MyDrive/transformer-scratch-yt/`.

In [ ]:
USE_DRIVE = False          # chapter 1 doesn't need Drive; training chapters will ask you to switch this on

import os
import numpy as np
import torch
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")          # opens Google's sign-in and permission prompt
    WORK = "/content/drive/MyDrive/transformer-scratch-yt"
else:
    WORK = "/content/transformer-scratch-yt"   # temporary: deleted when the session ends
for sub in ["data", "checkpoints", "runs"]:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

tok = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
print("GPT-2 small loaded:", f"{sum(p.numel() for p in model.parameters()):,}", "parameters · files go to", WORK)

---
## Chapter 1 · Text → numbers → vectors
You'll turn **"The cat sat on"** into the exact table of numbers GPT-2 sends into its first layer, using plain numpy:
token IDs, the embedding table, positions, batches, and how the lookup works as math. Needs only **Setup**.

### 1.0 · Numpy warm-up
A numpy **array** is a table of numbers. `.shape` tells you its size as `(rows, columns)`.

In [ ]:
import numpy as np

A = np.array([[1, 2, 0],
              [0, 1, 3]])
print(A.shape)      # (2, 3): 2 rows, 3 columns
print(A[1])         # row 1 (counting starts at 0): [0 1 3]
print(A[[1, 0]])    # rows 1 and 0, in that order
print(A + A)        # adding works number by number

**Matrix multiplication** uses `@`. Shape rule: `[a, b] @ [b, c] → [a, c]`.

In [ ]:
B = np.array([[2, 1],
              [0, 1],
              [1, 4]])
C = A @ B
print(C.shape)   # (2, 2)
print(C)         # top-left = 1*2 + 2*0 + 0*1 = 2

✏️ **YOUR TURN:** compute the top-left number of `C` by hand with a dot product: row 0 of `A` times column 0 of `B`, pair by pair, then added up. (Hint: `B[:, 0]` is column 0, and `np.sum(...)` adds things up.)

In [ ]:
top_left = ___

In [ ]:
assert top_left == C[0, 0]; print("✅ that's exactly what @ does for every row and column")

### 1.1 · Look inside GPT-2 small
Setup already loaded the real model as `model`. We only take two tables out of it.

In [ ]:
wte = model.transformer.wte.weight.detach().numpy()   # token embedding table
wpe = model.transformer.wpe.weight.detach().numpy()   # position table
print("wte", wte.shape, " wpe", wpe.shape)

Before running the next cell, predict: how many numbers are in `wte`? (rows × columns)

In [ ]:
print(f"{wte.size:,} numbers in the token table")
print(f"{sum(p.numel() for p in model.parameters()):,} numbers in the whole model")

### 1.2 · Text → token IDs
✏️ **YOUR TURN:** use the tokenizer to get the IDs. (Hint: `tok(text)["input_ids"]`.)

In [ ]:
text = "The cat sat on"
ids = ___
print(ids, [tok.decode([i]) for i in ids])

In [ ]:
assert ids == [464, 3797, 3332, 319]; print("✅ 4 tokens. Note the space at the front of ' cat'")

### 1.3 · IDs → rows of the table
✏️ **YOUR TURN:** pick the 4 rows of `wte` for our IDs, in order. The answer should have shape `(4, 768)`.

In [ ]:
tok_rows = ___
print(tok_rows.shape)
print(tok_rows[:, :4].round(4))   # first 4 numbers of each row

In [ ]:
assert tok_rows.shape == (4, 768) and np.allclose(tok_rows[1], wte[3797]); print("✅ one row per token: [4] → [4, 768]")

### 1.4 · The same lookup, as a matrix multiplication
A **one-hot** row is all zeros except a 1 at the ID's position. One-hot `[4, 50257]` @ table `[50257, 768]` → `[4, 768]`.

✏️ **YOUR TURN:** fill in the 1s, then multiply.

In [ ]:
onehot = np.zeros((4, 50257), dtype=np.float32)
for row, i in enumerate(ids):
    onehot[row, ___] = 1.0

via_matmul = ___
print(via_matmul.shape)

In [ ]:
assert np.array_equal(via_matmul, tok_rows); print("✅ identical: a lookup is a multiplication by one-hot rows")

### 1.5 · Add the position rows
✏️ **YOUR TURN:** take rows 0 to 3 of `wpe` (one per position) and add them to `tok_rows`.

In [ ]:
pos_rows = ___
x = ___
print(x.shape)

Now the real test: ask GPT-2 what it sends into layer 1 (`hidden_states[0]`) and compare.

In [ ]:
with torch.no_grad():
    out = model(torch.tensor([ids]), output_hidden_states=True)
gpt2_x = out.hidden_states[0][0].numpy()
diff = np.abs(x - gpt2_x).max()
print("largest difference:", diff)
assert diff < 1e-6; print("✅ your [4, 768] table is exactly what GPT-2 uses")

### 1.6 · Word order is now in the numbers
Same word, different position → different row. Try it:

In [ ]:
cat_at_1 = wte[3797] + wpe[1]
cat_at_50 = wte[3797] + wpe[50]
print("same token row, different sums:", np.abs(cat_at_1 - cat_at_50).max().round(4))

### 1.7 · Explore: which words are close?
**Cosine similarity** compares directions: 1 = same direction, 0 = no shared direction, negative = opposite.

✏️ **YOUR TURN:** finish the function. (Hint: `np.linalg.norm(v)` is a vector's length.)

In [ ]:
def nearest(word, k=8):
    i = tok(word)["input_ids"]
    assert len(i) == 1, "pick a word that is a single token (try a leading space)"
    v = wte[i[0]]
    sims = (wte @ v) / (np.linalg.norm(wte, axis=1) * ___)
    best = np.argsort(-sims)[1:k+1]
    return [(tok.decode([j]), round(float(sims[j]), 3)) for j in best]

nearest(" cat")

In [ ]:
assert nearest(" cat")[0][0] == " Cat"; print("✅ now try your own:", nearest(" king"))

### 1.8 · Bonus: what the position rows look like
Neighbouring positions should look alike.

In [ ]:
import matplotlib.pyplot as plt
P = wpe[:64] / np.linalg.norm(wpe[:64], axis=1, keepdims=True)
plt.imshow(P @ P.T, cmap="magma"); plt.colorbar(); plt.title("cosine similarity of positions 0-63"); plt.show()
print("length of row 0:", np.linalg.norm(wpe[0]).round(3), " row 1:", np.linalg.norm(wpe[1]).round(3))

### 1.9 · Many texts at once: a batch
A batch is a table of IDs with shape `[B, T]`: B rows, T positions. Texts have different lengths, so the simplest way to run prompts together is to **pad** the short ones and keep an **attention mask** (1 = real, 0 = pad). GPT-2 has no pad token, so we reuse end-of-text (ID 50256).

✏️ **YOUR TURN:** fill the `ids` and `mask` tables.

In [ ]:
prompts = ["The cat sat on", "My dog", "Once upon a time"]
ragged = [tok(p)["input_ids"] for p in prompts]
print(ragged)                     # lists of different lengths

B, T = len(ragged), max(len(r) for r in ragged)
ids_b = np.full((B, T), 50256)    # start with every cell = pad
mask = np.zeros((B, T), dtype=int)
for b, r in enumerate(ragged):
    ids_b[b, :len(r)] = ___
    mask[b, :len(r)] = ___
print(ids_b); print(mask)

In [ ]:
assert ids_b[1].tolist() == [3666, 3290, 50256, 50256] and mask[1].tolist() == [1, 1, 0, 0]; print("✅ a padded [B, T] batch with its mask")

✏️ **YOUR TURN:** embed the whole batch in one line each. `wte[ids_b]` works on a 2-D table of IDs and gives `[B, T, 768]`.
Then add the position rows `[T, 768]`: numpy copies them onto every text for you. That's **broadcasting**.

In [ ]:
X_b = ___                 # token rows for the whole batch
X_b = X_b + ___           # + the position rows for positions 0..T-1
print(X_b.shape)

In [ ]:
with torch.no_grad():
    hs = model(torch.tensor(ids_b), attention_mask=torch.tensor(mask), output_hidden_states=True).hidden_states[0].numpy()
real = mask == 1
print("largest difference on real tokens:", np.abs(X_b - hs)[real].max())
alone = wte[ragged[1]] + wpe[:len(ragged[1])]   # "My dog", the padded one
assert X_b.shape == (3, 4, 768) and np.abs(X_b - hs)[real].max() < 1e-6 and np.allclose(X_b[1, :2], alone)
print("✅ [3, 4] → [3, 4, 768], and row 1 (My dog) inside the batch = run alone")

### 1.10 · Training batches: pack, don't pad
Pretraining joins documents with an end-of-text token and cuts the stream into equal rows, so there's no padding. (Chat fine-tuning often does pad: Episode 6.)

✏️ **YOUR TURN:** cut `stream` into rows of `T_train` tokens (drop the leftover).

In [ ]:
docs = ["The cat sat on the mat.", "Paris is the capital of France.", "Water boils at 100 degrees."]
stream = []
for d in docs:
    stream += tok(d)["input_ids"] + [50256]
print(len(stream), "tokens in the stream")

T_train = 6
n_rows = len(stream) // T_train
rows = np.array(stream[:___]).reshape(n_rows, T_train)
for r in rows:
    print([tok.decode([i]) for i in r])

In [ ]:
assert rows.shape == (3, 6) and rows[1, 1] == 50256; print("✅ packed [3, 6]. Notice row 1 starts mid-sentence and holds two documents")

### 1.11 · Matrix multiplication with 3 dimensions
Rule: `[B, T, K] @ [K, N] = [B, T, N]`. Every sheet is multiplied by the **same** W.

✏️ **YOUR TURN:** compute `Y` three ways and check they agree.

In [ ]:
X = np.array([[[1, 0, 2, -1], [0, 1, 1, 2], [2, -1, 0, 1]],
              [[-1, 2, 0, 1], [1, 1, -1, 0], [0, 2, 1, -2]]])   # [2, 3, 4]
W = np.array([[1, 0], [2, 1], [0, -1], [1, 2]])                  # [4, 2]

Y_direct = ___                                       # just use @
Y_loop   = np.stack([___ for b in range(2)])         # sheet by sheet
Y_flat   = (X.reshape(6, 4) @ W).reshape(___)        # flatten, multiply, cut back
print(Y_direct.shape); print(Y_direct)

In [ ]:
assert np.array_equal(Y_direct, Y_loop) and np.array_equal(Y_direct, Y_flat) and Y_direct[0, 0, 0] == 0
print("✅ all three agree: the extra dimension just comes along")

### 1.12 · The lookup, by hand
The whole embedding step is one formula. For every batch row `b`, position `t` and column `d`:

`x[b, t, d] = wte[ ids[b, t], d ] + wpe[ t, d ]`

The ID is used as a **row number**, never multiplied. The position table is indexed by `t` only.

✏️ **YOUR TURN:** write the formula inside the three loops, on a tiny made-up model (5 tokens, rows of 3 numbers).

In [ ]:
E = np.array([[2, 0, 1], [1, 3, 0], [0, 1, 4], [3, 2, 2], [1, 1, 0]])   # token table [5, 3]
P = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 1]])                         # position table [3, 3]
ids_toy = np.array([[2, 4], [3, 2]])                                     # [B, T] = [2, 2]

B_, T_ = ids_toy.shape
x_loop = np.zeros((B_, T_, 3), dtype=int)
for b in range(B_):
    for t in range(T_):
        for d in range(3):
            x_loop[b, t, d] = ___
print(x_loop)

In [ ]:
x_fast = E[ids_toy] + P[:T_]                      # lookup + broadcasting, what numpy does in one line
x_onehot = np.eye(5, dtype=int)[ids_toy] @ E + P[:T_]   # one-hot [2, 2, 5] @ [5, 3]
assert np.array_equal(x_loop, x_fast) and np.array_equal(x_loop, x_onehot) and x_loop[0, 0].tolist() == [0, 2, 4]
print("✅ loops, lookup and one-hot all give the same [2, 2, 3]")

**Positions are looked up by position number, not by token ID.** The position numbers only depend on the length T.

✏️ **YOUR TURN:** make the position numbers for our batch and look them up.

In [ ]:
positions = ___            # the numbers 0, 1, ..., T-1 for our batch from 1.9 (hint: np.arange)
pos_rows_b = wpe[positions]
print(positions, pos_rows_b.shape)

In [ ]:
assert positions.tolist() == [0, 1, 2, 3] and np.array_equal(pos_rows_b, wpe[:4])
print("✅ [T] position numbers → [T, 768] rows, shared by every row of the batch")

**What GPT-2's own code does:** `nn.Embedding`, an index lookup. It never builds one-hot rows, but it would get the same numbers.

In [ ]:
import torch.nn.functional as F
W = model.transformer.wte.weight
ids_t = torch.tensor(ids_b)
with torch.no_grad():
    a = model.transformer.wte(ids_t)                       # what GPT-2 runs
    b = W[ids_t]                                           # plain indexing
    c = F.one_hot(ids_t, 50257).to(W.dtype) @ W            # one-hot [3, 4, 50257] @ [50257, 768]
print(a.shape, torch.equal(a, b), torch.equal(a, c))
print(f"one-hot costs {50257 * 768:,} multiply-adds per token; the lookup copies 768 numbers")

### 1.13 · Chapter 1 toolkit
Later chapters start here. Run this cell to get chapter 1's result without redoing the exercises.

In [ ]:
def embed(ids_b):
    """Token IDs [B, T] (or [T]) -> GPT-2's input to layer 1, [B, T, 768] (or [T, 768])."""
    ids_b = np.asarray(ids_b)
    T = ids_b.shape[-1]
    assert T <= wpe.shape[0], "GPT-2 has position rows for 1,024 tokens only"
    return wte[ids_b] + wpe[:T]

wte = model.transformer.wte.weight.detach().numpy()
wpe = model.transformer.wpe.weight.detach().numpy()
print(embed([[464, 3797, 3332, 319]]).shape)

#### Chapter 1 solutions
```python
top_left   = np.sum(A[0] * B[:, 0])
ids        = tok(text)["input_ids"]
tok_rows   = wte[ids]
onehot[row, i] = 1.0
via_matmul = onehot @ wte
pos_rows   = wpe[:4]          # or wpe[np.arange(len(ids))]
x          = tok_rows + pos_rows
# nearest(): ... * np.linalg.norm(v)
ids_b[b, :len(r)] = r
mask[b, :len(r)]  = 1
X_b = wte[ids_b]
X_b = X_b + wpe[:T]
rows = np.array(stream[:n_rows * T_train]).reshape(n_rows, T_train)
Y_direct = X @ W
Y_loop = np.stack([X[b] @ W for b in range(2)])
Y_flat = (X.reshape(6, 4) @ W).reshape(2, 3, 2)
x_loop[b, t, d] = E[ids_toy[b, t], d] + P[t, d]
positions = np.arange(ids_b.shape[1])
```
**Next chapter:** these rows start talking to each other: attention by hand, with the causal mask.